# Transcript Theme-Relevance Analyzer — Colab / Kaggle runner

This notebook clones [transcript-theme-analyzer](https://github.com/Samuel-Effiong/transcript-theme-analyzer) — a public GitHub repo —
then installs dependencies, takes your API key, takes your transcripts, runs
the analysis, and shows the results. Since it clones instead of embedding
the code, re-running the clone cell always pulls whatever's currently on
GitHub — no manual re-syncing needed after you push local edits.

> **Before you run this: a compute-expectation note.** This pipeline is
> **I/O-bound** — every step is a network call to an LLM API (OpenAI or
> OpenRouter). There's no local heavy computation, so Colab/Kaggle's GPU/CPU
> power won't make it faster; the bottleneck is the LLM provider's response
> time and any rate limits, not your local machine. What this notebook *does*
> give you: a free hosted environment, zero local Python/venv setup, and
> convenient built-in secrets management for your API key.


## Get the code

In [ ]:
import os

REPO_DIR = "transcript-theme-analyzer"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/Samuel-Effiong/transcript-theme-analyzer.git

%cd {REPO_DIR}


## Install dependencies

In [ ]:
!pip install -q "openai>=1.50.0" "pydantic>=2.6.0" "python-docx>=1.1.0"


## Configure your LLM provider + API key

This cell auto-detects where it's running:
- **Colab** — reads from Colab's Secrets manager (the key icon in the left
  sidebar). Add a secret named `OPENROUTER_API_KEY` there first, and make
  sure "Notebook access" is toggled on for it.
- **Kaggle** — reads from Kaggle's Secrets add-on (Add-ons → Secrets). Add a
  secret named `OPENROUTER_API_KEY` there first.
- **Neither found** — falls back to a masked prompt (`getpass`) so it still
  works, just not persisted anywhere.

Switching to OpenAI direct instead of OpenRouter: change the three
`LLM_*`/`OPENAI_API_KEY` lines below to match `.env.example` in the repo.


In [ ]:
import getpass
import os


def set_secret_env(env_var: str, prompt: str | None = None) -> None:
    if os.environ.get(env_var):
        return

    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(env_var)
    except Exception:
        pass

    if not value:
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore
            value = UserSecretsClient().get_secret(env_var)
        except Exception:
            pass

    if not value:
        value = getpass.getpass(prompt or f"Enter {env_var}: ")

    os.environ[env_var] = value


set_secret_env("OPENROUTER_API_KEY")

os.environ.setdefault("LLM_PROVIDER", "openrouter")
os.environ.setdefault("LLM_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("LLM_DEFAULT_MODEL", "anthropic/claude-sonnet-5")
os.environ.setdefault("CHUNK_SIZE_TOKENS", "12000")
os.environ.setdefault("CHUNK_OVERLAP_TOKENS", "800")
os.environ.setdefault("SINGLE_PASS_TOKEN_LIMIT", "20000")
os.environ.setdefault("MAX_CONCURRENT_CHUNKS", "8")
os.environ.setdefault("LLM_MAX_RETRIES", "5")

print("LLM_PROVIDER:", os.environ["LLM_PROVIDER"])
print("LLM_DEFAULT_MODEL:", os.environ["LLM_DEFAULT_MODEL"])
print("API key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


## ⚠️ Kaggle only: enable internet access

Kaggle notebooks have internet access **off by default**. Since every
analysis call needs to reach the LLM API, you must turn it on before
running: **Settings (right sidebar) → Internet → On** (requires phone
verification on your Kaggle account, one-time). Colab has internet on by
default — nothing to do there.


## Get your transcripts onto the notebook — Colab

Run the cell below **on Colab**. It uploads files directly into a local
`transcripts/` folder via a browser file picker. For a folder that persists
across sessions instead, mount Google Drive (commented out below) and point
`TRANSCRIPT_DIR` at a path under `/content/drive/MyDrive/...`.


In [ ]:
import os

TRANSCRIPT_DIR = "transcripts"

try:
    from google.colab import files  # type: ignore

    os.makedirs(TRANSCRIPT_DIR, exist_ok=True)
    print(f"Select your .txt/.docx transcript files (uploading into {TRANSCRIPT_DIR}/):")
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(TRANSCRIPT_DIR, name), "wb") as f:
            f.write(content)
    print(f"Uploaded {len(uploaded)} file(s) into {TRANSCRIPT_DIR}/")

    # Alternative: a persistent folder via Google Drive instead of a one-off upload.
    # from google.colab import drive
    # drive.mount("/content/drive")
    # TRANSCRIPT_DIR = "/content/drive/MyDrive/path/to/your/transcripts"
except ImportError:
    print("Not running on Colab — skipping this cell (see the Kaggle cell below instead).")


## Get your transcripts onto the notebook — Kaggle

On Kaggle: click **"Add Input"** (top right) → **Upload Dataset** → upload
your `.txt`/`.docx` transcript files. Once attached, they appear read-only
under `/kaggle/input/<your-dataset-name>/`. Set `KAGGLE_TRANSCRIPT_DIR`
below to match, and it'll override `TRANSCRIPT_DIR` from the Colab cell
above when running on Kaggle.


In [ ]:
import os

KAGGLE_TRANSCRIPT_DIR = "/kaggle/input/<your-dataset-name>"  # <-- change this

if os.path.isdir("/kaggle/input"):
    TRANSCRIPT_DIR = KAGGLE_TRANSCRIPT_DIR
    print(f"Running on Kaggle — TRANSCRIPT_DIR set to {TRANSCRIPT_DIR}")
else:
    print("Not running on Kaggle — leaving TRANSCRIPT_DIR as set by the Colab cell above.")


## Run the analysis

Edit `THEME` and `MODELS` below, then run this cell. It calls the same
`transcript_theme_analyzer.cli` you'd run locally — see the repo's README
for the full flag reference (`--glob`, multiple `--models`, etc.).


In [ ]:
THEME = "the glory of God"  # <-- change this to your search theme
MODELS = "anthropic/claude-sonnet-5"  # space-separated if running more than one model
OUT_DIR = "results"

!python -m transcript_theme_analyzer.cli --transcript-dir {TRANSCRIPT_DIR} --theme "{THEME}" --models {MODELS} --out-dir {OUT_DIR}


## View and download the results

Two final outputs land in `OUT_DIR`: `report.html` (rendered inline below)
and `report.docx`. On Colab, `report.docx` downloads directly. On Kaggle,
everything under `/kaggle/working` is automatically saved as notebook
output and downloadable from the Output tab after you save a version.


In [ ]:
from IPython.display import HTML, display

with open(f"{OUT_DIR}/report.html", encoding="utf-8") as f:
    display(HTML(f.read()))


In [ ]:
try:
    from google.colab import files  # type: ignore
    files.download(f"{OUT_DIR}/report.docx")
except ImportError:
    print(f"Not on Colab — find {OUT_DIR}/report.docx in the Output tab (Kaggle) "
          f"or the file browser on the left (other environments).")
